# Whisper Large v3 LoRA Levantine Custom Run: Palestine + Jordan + North Levantine

This notebook mirrors the practical structure of the original Whisper Medium mini run, but swaps in the exact Levantine datasets requested for training and held-out evaluation.

Custom dataset plan:
- train on the requested Casablanca Palestine/Jordan validation parquet files
- train on the requested Omnilingual APC North Levantine Arrow files (`data-00000`, `data-00001`)
- split the requested Casablanca test parquet files 50/50 into validation and test
- split Omnilingual `data-00002-of-00003.arrow` 50/50 into validation and test
- train for up to 50 epochs with early stopping patience `3` on rising validation loss


## 1. Config

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = Path("/home/MohammadNabulsi/whisper")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from Runs.whisper_large_v3_levantine_custom_streaming_5minckpt.pipeline import (
    NOTEBOOK_PATH,
    RUN_DIR,
    SMOKE_NOTEBOOK_PATH,
    config_snapshot,
    ensure_run_layout,
    make_config,
    setup_logging,
)

RUN_SMOKE_TEST = False
TRAIN_HOURS_CAP = 50.0
EVAL_SAMPLE_CAP = 100
RUN_BASELINE_BEFORE_TRAIN = False
RUN_POST_TRAIN_EVAL = True
RESUME_TRAINING = False
RESUME_FROM_CHECKPOINT = str(RUN_DIR / "checkpoints" / "checkpoint-388") if RESUME_TRAINING else None

config = make_config(
    smoke_mode=RUN_SMOKE_TEST,
    train_hours_cap=TRAIN_HOURS_CAP,
    eval_sample_cap=EVAL_SAMPLE_CAP,
    num_train_epochs=50,
    run_baseline_before_train=RUN_BASELINE_BEFORE_TRAIN,
    run_post_train_eval=RUN_POST_TRAIN_EVAL,
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT,
)
ensure_run_layout()
log_path = setup_logging()
print(json.dumps(config_snapshot(config), ensure_ascii=False, indent=2))
print("Notebook path:", SMOKE_NOTEBOOK_PATH if RUN_SMOKE_TEST else NOTEBOOK_PATH)
print("Log path:", log_path)


{
  "model_name": "openai/whisper-large-v3",
  "run_id": "whisper_large_v3_levantine_custom_streaming_5minckpt",
  "sample_rate": 16000,
  "min_audio_seconds": 0.3,
  "drop_audio_at_or_above_seconds": 30.0,
  "train_hours_cap": 50.0,
  "eval_sample_cap": 100,
  "train_batch_size": 4,
  "eval_batch_size": 4,
  "gradient_accumulation_steps": 4,
  "learning_rate": 0.0001,
  "warmup_ratio": 0.05,
  "weight_decay": 0.01,
  "max_grad_norm": 1.0,
  "num_train_epochs": 50,
  "logging_steps": 5,
  "checkpoint_every_minutes": 5,
  "save_total_limit": 6,
  "generation_max_new_tokens": 256,
  "early_stopping_patience": 3,
  "split_seed": 42,
  "train_dataloader_num_workers": 4,
  "language": "ar",
  "task": "transcribe",
  "smoke_mode": false,
  "run_baseline_before_train": false,
  "run_post_train_eval": true,
  "force_rebuild_manifests": false,
  "resume_from_checkpoint": null,
  "use_fp16": true,
  "lora_r": 16,
  "lora_alpha": 32,
  "lora_dropout": 0.05,
  "lora_target_modules": [
    "q_proj"

## 2. Build Filtered Mini Manifests

In [2]:
from Runs.whisper_large_v3_levantine_custom_streaming_5minckpt.pipeline import prepare_manifests

manifest_state = prepare_manifests(config)
selection_summary = manifest_state["selection_summary"]
print(json.dumps(selection_summary, ensure_ascii=False, indent=2))


2026-06-24 18:41:07,697 | INFO | whisper_large_v3_run | prepared split=train rows=1860 kept=1541 dropped_empty=0 dropped_duration=319
2026-06-24 18:41:07,740 | INFO | whisper_large_v3_run | prepared split=eval_pool rows=1515 kept=1514 dropped_empty=0 dropped_duration=1
2026-06-24 18:41:07,789 | INFO | whisper_large_v3_run | prepared split=eval_pool rows=172 kept=19 dropped_empty=0 dropped_duration=153


{
  "run_id": "whisper_large_v3_levantine_custom_streaming_5minckpt",
  "drop_segments_at_or_above_seconds": 30.0,
  "full_counts_after_filter": {
    "train": 1541,
    "eval_parquet_pool": 1514,
    "eval_arrow_pool": 19
  },
  "selected_counts": {
    "train": 1541,
    "val": 766,
    "test": 767
  },
  "selected_hours": {
    "train": 2.1407003992305276,
    "val": 1.0070916678090185,
    "test": 1.037511566857676
  },
  "train_by_source_group": {
    "casablanca_levantine_train": {
      "rows": 1514,
      "hours": 2.0004557985360836
    },
    "omnilingual_apc_north_levantine_train": {
      "rows": 27,
      "hours": 0.14024460069444444
    }
  },
  "val_by_source_group": {
    "casablanca_levantine_eval_pool": {
      "rows": 757,
      "hours": 0.9691543645682779
    },
    "omnilingual_apc_north_levantine_eval_pool": {
      "rows": 9,
      "hours": 0.037937303240740745
    }
  },
  "test_by_source_group": {
    "casablanca_levantine_eval_pool": {
      "rows": 757,
      

## 3. Baseline Whisper Medium Predictions

In [ ]:
from Runs.whisper_large_v3_levantine_custom_streaming_5minckpt.pipeline import load_whisper_bundle, run_predictions

baseline_bundle = load_whisper_bundle(config)
baseline_test_metrics = None
if config.run_baseline_before_train:
    baseline_test_metrics = run_predictions(
        manifest_state["test_rows"],
        config,
        baseline_bundle,
        name="baseline_test_predictions",
    )
print(json.dumps(baseline_test_metrics, ensure_ascii=False, indent=2))


## 4. Train LoRA Adapters

In [ ]:
from Runs.whisper_large_v3_levantine_custom_streaming_5minckpt.pipeline import train_model

training_summary = train_model(config, manifest_state)
print(json.dumps(training_summary, ensure_ascii=False, indent=2))


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/home/MohammadNabulsi/whisper/Runs/whisper_large_v3_levantine_custom_streaming_5minckpt/pipeline.py:1139: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


Epoch,Training Loss,Validation Loss


## 5. Tuned Validation and Test Predictions

In [ ]:
from pathlib import Path
import json

from Runs.whisper_large_v3_levantine_custom_streaming_5minckpt.pipeline import (
    BEST_MODEL_DIR,
    load_whisper_bundle,
    run_predictions,
)

adapter_path = Path(training_summary["best_checkpoint"])
if not (adapter_path / "adapter_config.json").exists():
    adapter_path = BEST_MODEL_DIR

tuned_bundle = load_whisper_bundle(config, adapter_path=adapter_path)

val_prediction_metrics = run_predictions(
    manifest_state["val_rows"],
    config,
    tuned_bundle,
    name="tuned_val_predictions",
) if config.run_post_train_eval else None

test_prediction_metrics = run_predictions(
    manifest_state["test_rows"],
    config,
    tuned_bundle,
    name="tuned_test_predictions",
) if config.run_post_train_eval else None

print("Validation metrics:")
print(json.dumps(val_prediction_metrics, ensure_ascii=False, indent=2))
print("Test metrics:")
print(json.dumps(test_prediction_metrics, ensure_ascii=False, indent=2))

## 6. Summary Report

In [ ]:
import json

from Runs.whisper_large_v3_levantine_custom_streaming_5minckpt.pipeline import write_summary_report

summary_report = write_summary_report(
    config,
    selection_summary,
    baseline_test_metrics if "baseline_test_metrics" in globals() else None,
    val_prediction_metrics if "val_prediction_metrics" in globals() else None,
    test_prediction_metrics if "test_prediction_metrics" in globals() else None,
    training_summary,
)
print(json.dumps(summary_report, ensure_ascii=False, indent=2))

## 7. Integrity Report

In [ ]:
import json

from Runs.whisper_large_v3_levantine_custom_streaming_5minckpt.pipeline import write_integrity_report

integrity_report = write_integrity_report(
    config,
    selection_summary,
    baseline_test_metrics if "baseline_test_metrics" in globals() else None,
    val_prediction_metrics if "val_prediction_metrics" in globals() else None,
    test_prediction_metrics if "test_prediction_metrics" in globals() else None,
    training_summary,
)
print(json.dumps(integrity_report, ensure_ascii=False, indent=2))

test_health = integrity_report.get("prediction_health", {}).get("test_predictions", {})
if test_health.get("object_dump_predictions", 0):
    raise RuntimeError("Decode guard failed: found object-dump predictions in test output.")
if test_health.get("empty_predictions", 0):
    raise RuntimeError("Integrity check failed: found empty predictions in test output.")

print("WHISPER MEDIUM MINI RUN INTEGRITY CHECK PASSED")